<a href="https://colab.research.google.com/github/almagomen/v2v/blob/opencut_launch/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video + OpenCut – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `FISH_API_KEY`, `GHP_GITHUB_TOKEN`
3. **Modelos en Drive**: Si tu carpeta `v2v` está en **Compartido conmigo** en Drive, hazle clic derecho ➔ *Organizar* ➔ *Añadir acceso directo* ➔ elige **Mi unidad**.
4. Ejecuta la celda para iniciar FastAPI y el editor OpenCut.

In [ ]:
import os
import sys
import time
import threading
import subprocess
import warnings
from pathlib import Path
from google.colab import drive, userdata, output
import uvicorn
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIGURACIÓN ---
REPO_PATH    = Path("/content/v2v")
BRANCH_NAME  = "opencut_launch"
BACKEND_PORT = 8000
FRONTEND_PORT = 5173
LOG_FILE     = Path("/content/v2v.log")

# --- 1. GOOGLE DRIVE Y SECRETS ---
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
if not os.path.exists("/content/drive/MyDrive/v2v"):
    print("💡 Recordatorio: Si tu carpeta 'v2v' está en 'Compartido conmigo', añade un acceso directo a 'Mi unidad' en Google Drive.")

# Exportar Secrets al entorno para subprocesos (FastAPI/Uvicorn)
for key in ["GOOGLE_API_KEY", "FISH_API_KEY", "GHP_GITHUB_TOKEN"]:
    try:
        val = userdata.get(key)
        if val: os.environ[key] = val
    except Exception: pass
os.environ["DRIVE_MOUNT"] = "true"
os.environ["V2V_BASE_DIR"] = "/content"

get_ghp = os.environ.get("GHP_GITHUB_TOKEN", "")
repo_url = f"https://{get_ghp}@github.com/almagomen/v2v.git"

# --- 2. GESTIÓN DEL REPOSITORIO ---
if not REPO_PATH.exists():
    !git clone -q -b {BRANCH_NAME} {repo_url} {REPO_PATH}
    %cd {REPO_PATH}
else:
    %cd {REPO_PATH}
    !git fetch origin {BRANCH_NAME} -q
    # Mostrar qué cambió antes de actualizar
    diff_output = !git diff --name-status HEAD origin/{BRANCH_NAME}
    info_commit = !git log origin/{BRANCH_NAME} --oneline -1
    tags = {'M': '✏️ Modificado', 'A': '✅ Añadido', 'D': '🗑️ Eliminado', 'R': '🔄 Renombrado'}
    if diff_output:
        print(f"\n🚀 ¡Cambios detectados!\n📌 Commit: {info_commit[0]}.")
        for line in diff_output:
            s, f = line.split(None, 1)
            print(f"   • {tags.get(s[0], '❓ ' + s[0])}: {f}")
        !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q
    else:
        !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q
        print("✔️ Repositorio al día.")

if str(REPO_PATH) not in sys.path: sys.path.insert(0, str(REPO_PATH))

# --- 3. INSTALACIÓN QUIRÚRGICA (sin romper NumPy/PyTorch nativos de Colab) ---
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias de Python y FastAPI...")
    !uv pip install fastapi uvicorn websockets python-multipart google-genai -q

    print("⚡ Instalando Deno (para resolver scripts de YouTube en yt-dlp)...")
    !curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1
    os.environ["PATH"] += ":/root/.deno/bin"

    # Instalar librerías de ML sin tocar versiones de torch/numpy/scipy de Colab
    # --no-deps donde no queremos que pip resuelva dependencias upstream
    !uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install "yt-dlp[default]" bs-roformer-infer tinydb sentencex torchcodec==0.7.0 'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    !uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' -q

    print("⚡ Instalando Bun para OpenCut...")
    !curl -fsSL https://bun.sh/install | bash -s "bun-v1.2.18" -q
    os.environ["PATH"] += ":/root/.bun/bin"
    %cd /content/v2v/opencut/apps/web
    !bun install --no-save
    %cd /content/v2v

    deps_installed = True

# --- 4. CARGA PERSISTENTE DE MODELOS IA (solo una vez por sesión de GPU) ---
if 'models_loaded' not in globals():
    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model

    # Guardar instancias reales en globals para inyectarlas al backend
    # Esto evita recargar pesos desde disco en cada reinicio del servidor
    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': get_roformer_model(),
        'models_loaded': True
    })
    print("✅ Modelos listos en RAM/GPU.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 5. CONFIGURACIÓN DE LOGS ---
log_config = {
    "version": 1, "disable_existing_loggers": False,
    "formatters": {
        "default": {"()": "uvicorn.logging.DefaultFormatter", "fmt": "%(levelprefix)s %(message)s", "use_colors": True},
        "access": {"()": "uvicorn.logging.AccessFormatter", "fmt": '%(levelprefix)s %(client_addr)s - "%(request_line)s" %(status_code)s', "use_colors": True},
    },
    "handlers": {
        "console": {"class": "logging.StreamHandler", "formatter": "default", "stream": "ext://sys.stdout"},
        "file": {"class": "logging.FileHandler", "formatter": "default", "filename": str(LOG_FILE), "mode": "w"},
        "access_console": {"class": "logging.StreamHandler", "formatter": "access", "stream": "ext://sys.stdout"},
    },
    "loggers": {
        "uvicorn": {"handlers": ["console", "file"], "level": "INFO", "propagate": False},
        "uvicorn.error": {"handlers": ["file"], "level": "INFO"},
        "uvicorn.access": {"handlers": ["access_console", "file"], "level": "INFO", "propagate": False},
    },
}

# --- 6. SERVIDOR BACKEND (Con inyección de modelos pre-cargados) ---
class ControlledServer(uvicorn.Server):
    """Uvicorn sin signal handlers para poder correr en hilo dentro de Colab."""
    def install_signal_handlers(self): return

def run_backend():
    global server
    # Limpiar módulos de la app (sin tocar los pesados de modelos)
    for m in [m for m in sys.modules if m.startswith(('api', 'src'))]:
        if 'models' in m: continue  # Protegemos referencias a modelos
        del sys.modules[m]

    from api.server import app

    # Inyectar los modelos pre-cargados en el estado de la APP
    # Los endpoints los leerán de app.state en lugar de llamar a load_model()
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_config=log_config, loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config)
    server.run()

# Reinicio limpio del backend si ya estaba corriendo
if 'server' in globals() and server.started:
    print("🛑 Reiniciando backend..."); server.should_exit = True; time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# --- 7. FRONTEND OPENCUT (Vite via Bun) ---
OPENCUT_PATH = REPO_PATH / "opencut" / "apps" / "web"
print("🏗️ Preparando OpenCut (frontend Vite)...")
!fuser -k {FRONTEND_PORT}/tcp 2>/dev/null
f_env = os.environ.copy(); f_env["FORCE_COLOR"] = "1"
process = subprocess.Popen(
    ["bun", "run", "dev", "--", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
    cwd=str(OPENCUT_PATH), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=f_env
)

def log_reader(pipe):
    C, RS = "\033[1;36m", "\033[0m"
    with open(LOG_FILE, "a") as f:
        for line in iter(pipe.readline, ''):
            if line.strip():
                f.write(f"\n{C}[FRONTEND]{RS} {line.strip()}\n"); f.flush()

threading.Thread(target=log_reader, args=(process.stdout,), daemon=True).start()
time.sleep(3)  # Dar tiempo a Vite para arrancar

# --- 8. TÚNEL NATIVO DE COLAB (sin ventana de advertencia) ---
print("\n✨ ENTORNO LISTO ✨")
output.serve_kernel_port_as_window(FRONTEND_PORT, path="/", anchor_text="🚀 Abrir Editor OpenCut + V2V", skip_warning=True)
